# Workflow: using an abstract model as a shield

**Motivation**

**What do we show here?**

**Disclaimer**

## Imports and helpers    

In [1]:
import gymnasium as gym
import numpy as np

from verigym.environments.generativeenv import GenerativeEnv
from verigym.environments.labeling import StateLabel, AbstractStateLabeler
from verigym.abstraction.learn_abstraction import create_abstraction
from verigym.policy.policy import RandomizedPolicy
from verigym.frameworks.stormpy.stormpy_utils import build_stormpy_mdp

## The actual Workflow

First, we choose our environment and load it into a `VeriGym``GenerativeEnv`.
We use `LunarLander-v3`as an example environment here.

In [2]:
env = GenerativeEnv.from_gymnasium(
    gym.make("LunarLander-v3")
)

We now define a label that declares states that are too far outside the landing pad as "unsafe" using the `StateLabel` class.
The x dimension of a state is in [-2.5, 2.5]. We consider states that are further out than [-1, 1] to be unsafe.

In [3]:
unsafe_label = StateLabel("unsafe",
                          lambda s:
                          (s[0] <= -1.0) | (s[0] >= 1.0)
                          )
env.add_state_label(unsafe_label)

Now, we need to discretize the environment and learn an approximation of its transition and reward functions.

In [4]:
learned_env = create_abstraction(
    env,
    exploration_policy=RandomizedPolicy(env),
    num_steps=int(1e6),
    bin_edges_per_state_dim=np.array([11, 11, 6, 6, 6, 6, 2, 2], dtype=int),
    bin_edges_per_action_dim=4
)

Simulation time: 22.0756s
Trajectories in dataset: 10777
processing in  17.821619987487793
aggregating..
aggregating in 0.007524728775024414
Learning Abstraction: 17.8469s


In [5]:
print(learned_env.nr_states, learned_env.nr_actions)
print(learned_env.transition_function.get_sparsity())
print(learned_env.observation_space)

627264 4
2.842092316271957e-09
Discrete(627264)


In [6]:
abstract_state_labeler = AbstractStateLabeler(env.state_labeler,
                                              learned_env.abstraction_map)
learned_env.state_labeler = abstract_state_labeler

In [8]:
learned_env.abstraction_map.abstract_to_original_state(0)

array([ -2.5      ,  -2.5      , -10.       , -10.       ,  -6.2831855,
       -10.       ,   0.       ,   0.       ], dtype=float32)

In [7]:
propertystr = 'Pmin=? [F "unsafe"]'
mdp = build_stormpy_mdp(learned_env)

TypeError: object of type 'numpy.float32' has no len()